Before runninng this lab, create and activate venv. This lab works with python 3.11+

In [7]:
#python -m venv venv
#source venv/bin/activate

In [4]:
%pip install langchain langchain-core langchain-community langchain-classic langchain-chroma langchain-ollama langchain-text-splitters chromadb beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import RetrievalQA
import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_text_splitters import HTMLHeaderTextSplitter

from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import RetrievalQA
import re
import bs4


/var/folders/hj/s2sl4h3j275blqcx4kf042_r0000gp/T/ipykernel_64510/1762360570.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# 0. Clean data
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

def extract_page(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    main = soup.find("main")

    if main is None:
        raise ValueError(f"Main content not found: {url}")

    # Remove irrelevant elements
    for element in main.select(
        "nav, header, footer, aside, script, style"
    ):
        element.decompose()

    # Extract cards first
    card_documents = []

    for heading in main.select(
        '[data-component-part="card-title"]'
    ):
        card = heading.parent

        content = card.select_one(
            '[data-component-part="card-content"]'
        )

        if content is None:
            continue

        title = heading.get_text(" ", strip=True)
        body = content.get_text(" ", strip=True)

        if body:
            card_documents.append(
                Document(
                    page_content=f"{title}\n\n{body}",
                    metadata={
                        "section": title,
                        "source": url,
                        "content_type": "card"
                    }
                )
            )

        # Remove the extracted card to prevent duplication
        card.decompose()

    # Extract remaining documentation sections
    section_documents = html_splitter.split_text(str(main))

    for doc in section_documents:
        doc.metadata["source"] = url
        doc.metadata["content_type"] = "section"

    return card_documents + section_documents

In [5]:
# Get multiple urls
urls = [
    "https://docs.langchain.com/oss/python/langchain/overview",
    "https://docs.langchain.com/oss/python/langchain/retrieval",
    "https://docs.langchain.com/oss/python/langchain/agents",
]

loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/LangChain",)
)

# 1. Load and clean the document about AI
documents = loader.load()

for url in urls:
    page_docs = extract_page(url)
    documents.extend(page_docs)

# 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

# 3. Set up the embedding model. (Use an embedding model to create vector representations.)
embedding_model = OllamaEmbeddings(model="nomic-embed-text")


# 4. Create a vector store
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="langchain_improved_v3",
    persist_directory="./chroma_improved_v3"
)

# 5. Create a retriever
retriever = vector_store.as_retriever()

# 6. Define a function to search for relevant information
def search_documents(query, top_k=3):
    """Search for documents relevant to a query"""
    # Use the retriever to get relevant documents
    docs = retriever.invoke(query)
    
    # Limit to top_k if specified
    return docs[:top_k]

# 7. Test with a few queries
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Difference between Lanngchain and Langgraph?"
]



for query in test_queries:
    results = search_documents(query, 3)

    print(f"\nQuery: {query}")
    print(f"Retrieved {len(results)} documents")

    for i, doc in enumerate(results, start=1):
        clean_text = re.sub(r'\s+', ' ', doc.page_content).strip()

        print(f"\n{i}. {clean_text[:500]}...")
   
# for i, doc in enumerate(results, start=1):
#     print(f"Document {i}")
#     print("Length:", len(doc.page_content))
#     print("Metadata:", doc.metadata)
#     print("Preview:", repr(doc.page_content[:150]))
#     print("-" * 30)



Query: What is LangChain?
Retrieved 3 documents

1. Free and open-source software portal LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.[2] History[edit]...

2. Built on top of LangGraph LangChain’s agents are built on top of LangGraph. This allows us to take advantage of LangGraph’s durable execution, human-in-the-loop support, persistence, and more....

3. LangChain tools[edit]...

Query: How do retrievers work?
Retrieved 3 documents

1. Retrievers A retriever is an interface that returns documents given an unstructured query....

2. Agentic Retrieval-Augmented Generation (RAG) when how The only thing an agent needs to enable RAG behavior is access to one or more that can fetch external knowledge, such as document

In [ ]:
# 8. Similarity score
results = vector_store.similarity_search_with_score(
    "How do retrievers work?",
    k=3
)

for doc, score in results:
    print("\nScore:", score)
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:500])

In [3]:
print("Total chunks:", len(chunks))
print("Unique chunks:", len(set(chunk.page_content for chunk in chunks)))

Total chunks: 219
Unique chunks: 193
